Random Forest Regressor for predicting missing values

In [20]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error,r2_score,mean_absolute_error, mean_absolute_percentage_error,mean_squared_error
from sklearn.impute import SimpleImputer
import seaborn as sns
import numpy as np
import pandas as pd

df = sns.load_dataset('titanic')
df.isnull().sum().sort_values(ascending=False)

deck           688
age            177
embarked         2
embark_town      2
sex              0
pclass           0
survived         0
fare             0
parch            0
sibsp            0
class            0
adult_male       0
who              0
alive            0
alone            0
dtype: int64

remove deck column

In [21]:
# remove deck column 
df.drop('deck',axis=1, inplace=True)

df.isnull().sum().sort_values(ascending=False)

age            177
embarked         2
embark_town      2
sex              0
pclass           0
survived         0
parch            0
sibsp            0
class            0
fare             0
who              0
adult_male       0
alive            0
alone            0
dtype: int64

In [22]:
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,Southampton,no,True


important point is always encode the data in any algorithm

In [23]:
# LabelEncoder is used to convert the catagrical values into numeric in the form of 0 1 2 0 1 2 0 1 2 
from sklearn.preprocessing import LabelEncoder
# mention the columns the to encode 
columns_to_encode = ['sex','embarked','who','class','embark_town','alive']
# Dictionary to store labelEncoders for each column
label_encoders = {}
# Loop to apply LabelEncoder to impute each column 
for col in columns_to_encode:
    # call the LabelEncoder and store in an object
    le = LabelEncoder()
    # train the model and tranform
    df[col] = le.fit_transform(df[col])
    # store the encoder into the label_encoders dictionary
    label_encoders[col] = le
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone
0,0,3,1,22.0,1,0,7.2500,2,2,1,True,2,0,False
1,1,1,0,38.0,1,0,71.2833,0,0,2,False,0,1,False
2,1,3,0,26.0,0,0,7.9250,2,2,2,False,2,1,True
3,1,1,0,35.0,1,0,53.1000,2,0,2,False,2,1,False
4,0,3,1,35.0,0,0,8.0500,2,2,1,True,2,0,True


We have to first impute the missing values in the age column before we can use it to predict the missing values in the embarked and embark_town columns.

In [24]:
# split the dataset into two parts one with the missing values, one without 
df_with_missing = df[df['age'].isnull()]
# dropna removes all the rows with missing values 
df_without_missing = df[df['age'].notnull()]

Let's see the shape of the dataset with and without the missing values

In [25]:
print("The shape of the original dataset is L: ", df.shape)
print("The shape of the dataset with missing values removed is: ",df_without_missing.shape)
print("The shape of the dataset with the missing values is : ",df_with_missing.shape)

The shape of the original dataset is L:  (891, 14)
The shape of the dataset with missing values removed is:  (714, 14)
The shape of the dataset with the missing values is :  (177, 14)


Let's see the first five rows of the dataset with the missing values:

In [26]:
df_with_missing.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone
5,0,3,1,NaN,0,0,8.4583,1,2,1,True,1,0,True
17,1,2,1,NaN,0,0,13.0000,2,1,1,True,2,1,True
19,1,3,0,NaN,0,0,7.2250,0,2,2,False,0,1,True
26,0,3,1,NaN,0,0,7.2250,0,2,1,True,0,0,True
28,1,3,0,NaN,0,0,7.8792,1,2,2,False,1,1,True


Let's see the first five  rows of the dataset without the missing values

In [27]:
df_without_missing.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone
0,0,3,1,22.0,1,0,7.2500,2,2,1,True,2,0,False
1,1,1,0,38.0,1,0,71.2833,0,0,2,False,0,1,False
2,1,3,0,26.0,0,0,7.9250,2,2,2,False,2,1,True
3,1,1,0,35.0,1,0,53.1000,2,0,2,False,2,1,False
4,0,3,1,35.0,0,0,8.0500,2,2,1,True,2,0,True


Let's see the names of all the column in the dataset

In [28]:
print(df.columns)

Index(['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare',
       'embarked', 'class', 'who', 'adult_male', 'embark_town', 'alive',
       'alone'],
      dtype='object')


Apply Random forest Regressor model

In [29]:
# Features and target
X = df_without_missing.drop('age', axis=1)
y = df_without_missing['age']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# Random Forest Model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)

rf_model.fit(X_train, y_train)

# Prediction
y_pred = rf_model.predict(X_test)

# Evaluation

print("RMSE for Random Forest Imputation:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score for Random Forest Imputation : ",r2_score(y_test,y_pred))
print("MAE for Random Forest Imputation: ",mean_absolute_error(y_test,y_pred))
print("MAPE for Random Forest Imputation: ",mean_absolute_percentage_error(y_test,y_pred))


RMSE for Random Forest Imputation: 11.081260589808045
R2 Score for Random Forest Imputation :  0.33769388288226154
MAE for Random Forest Imputation:  8.666661815622195
MAPE for Random Forest Imputation:  0.40839466096086574


In [30]:
df_with_missing.isnull().sum().sort_values(ascending = False)

age            177
survived         0
pclass           0
sex              0
sibsp            0
parch            0
fare             0
embarked         0
class            0
who              0
adult_male       0
embark_town      0
alive            0
alone            0
dtype: int64

predict missing values 

In [36]:
# predict missing values 
y_pred = rf_model.predict(df_with_missing.drop(['age'],axis=1))


remove unnesecory warning 

In [32]:
# remove the warning 
import warnings
warnings.filterwarnings('ignore')

# replace the missing values with the predicted values 
df_with_missing['age']= y_pred

# check the missing values 
df_with_missing.isnull().sum().sort_values(ascending = False)

survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
class          0
who            0
adult_male     0
embark_town    0
alive          0
alone          0
dtype: int64

concatinate the two dataframes

In [33]:
# concatinate the two dataframes
df_complete = pd.concat([df_with_missing,df_without_missing],axis = 0)

# print the shape of the complete dataframe
print("The shape of the complete dataframe is : ",df_complete.shape)

# check the first 5 rows if the complete dataframe 
df_complete.head()

The shape of the complete dataframe is :  (891, 14)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone
5,0,3,1,32.976583,0,0,8.4583,1,2,1,True,1,0,True
17,1,2,1,35.642218,0,0,13.0000,2,1,1,True,2,1,True
19,1,3,0,18.347000,0,0,7.2250,0,2,2,False,0,1,True
26,0,3,1,35.571486,0,0,7.2250,0,2,1,True,0,0,True
28,1,3,0,20.651429,0,0,7.8792,1,2,2,False,1,1,True


Decode the encoded data

In [34]:
# inverse the transform for encoded columns 
for col in columns_to_encode:
    # Retrieve the corresponding LabelEncoder for  the column 
    le = label_encoders[col]
    # Inverse transform the data and convert to integer type
    df_complete[col] = le.inverse_transform(df[col])

df_complete.head()


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone
5,0,3,male,32.976583,0,0,8.4583,S,Third,man,True,Southampton,no,True
17,1,2,female,35.642218,0,0,13.0000,C,First,woman,True,Cherbourg,yes,True
19,1,3,female,18.347000,0,0,7.2250,S,Third,woman,False,Southampton,yes,True
26,0,3,female,35.571486,0,0,7.2250,S,First,woman,True,Southampton,yes,True
28,1,3,male,20.651429,0,0,7.8792,S,Third,man,False,Southampton,no,True
